# Generate HW2 Arduino Test Arrays

This helper notebook reproduces the HW2 preprocessing pipeline and prints the next 10 test samples, `X_test[5:15]`, and their labels, `y_test[5:15]`, as C arrays for the Arduino sketch.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import c_writer

column_names = [
    "duration", "protocoltype", "service", "flag", "srcbytes", "dstbytes",
    "land", "wrongfragment", "urgent", "hot", "numfailedlogins", "loggedin",
    "numcompromised", "rootshell", "suattempted", "numroot", "numfilecreations",
    "numshells", "numaccessfiles", "numoutboundcmds", "ishostlogin", "isguestlogin",
    "count", "srvcount", "serrorrate", "srvserrorrate", "rerrorrate", "srvrerrorrate",
    "samesrvrate", "diffsrvrate", "srvdiffhostrate", "dsthostcount", "dsthostsrvcount",
    "dsthostsamesrvrate", "dsthostdiffsrvrate", "dsthostsamesrcportrate",
    "dsthostsrvdiffhostrate", "dsthostserrorrate", "dsthostsrvserrorrate",
    "dsthostrerrorrate", "dsthostsrvrerrorrate", "attack", "lastflag"
]

# Load and preprocess the dataset exactly like the main HW2 notebook.
data = pd.read_csv("Network_anomaly_data.txt", sep=",", names=column_names)
data = data.drop(columns=["land", "urgent", "numfailedlogins", "numoutboundcmds"])
data["attack"] = np.where(data["attack"] == "normal", "normal", "attack")

for column in ["protocoltype", "service", "flag", "attack"]:
    encoder = LabelEncoder()
    data[column] = encoder.fit_transform(data[column])

X = data.drop(columns=["attack"]).values
y = data["attack"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Select test samples 5 through 14 for Question 5(d).
Xtest_next10 = X_test[5:15, :]
ytest_next10 = y_test[5:15]

print(c_writer.create_array(Xtest_next10, "float", "X_test"))
print(c_writer.create_array(ytest_next10.astype(np.uint8), "uint8_t", "y_test"))